# CMZ — Behavior detection results (day & night)

Proof that the deployed system works. For each behavior (**Eating, Locomotion, Resting**), in
**daylight and infrared**, it tracks the civet and labels its behavior, producing an **observation
sheet** (time → Active/Inactive → behavior) and an **annotated preview**.

**Pipeline:** custom YOLO-seg civet detector + ByteTrack tracker → 3-class ResNet behavior
classifier → motion gate + food-tray rule.

**Put these on Drive first (`CMZprojectphase2/`):**
- `models/civet_bowl_seg_best.pt` — the YOLO civet detector (rename `best.pt` to this)
- `models/model.pt` — the 3-class behavior ResNet
- `data/raw/cam3/A03_20260405140327.mp4` — daytime video
- `data/raw/cam3/A03_20260405000000.mp4` — night video

Run the two cells below top to bottom.

In [ ]:
# 1) Setup: mount Drive, pull the code, install deps
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/CMZprojectphase2")
assert PROJECT_ROOT.is_dir(), f"Add a shortcut to the shared {PROJECT_ROOT.name} under My Drive"
drive_root = PROJECT_ROOT

repo = Path("/content/cmz-project"); branch = "feat/sam3-pixel-annotation"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", f"origin/{branch}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", branch,
                    "https://github.com/Siwali/cmz-project.git", str(repo)], check=True)
PIPE = repo / "pipeline"; sys.path.insert(0, str(PIPE))

!pip -q install ultralytics imageio-ffmpeg
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## The showcase

Five short clips, one per behavior per lighting condition:

| clip | lighting | what it shows |
|------|----------|---------------|
| Locomotion_day | color | civet circling, away from the tray |
| Eating_day | color | civet head-down at the tray (tray rule) |
| Resting_day | color | civet lying still (motion gate keeps it Inactive) |
| Eating_night | infrared | civet eating at the tray — CNN reads the posture, no tray box (camera moved at night) |
| Locomotion_night | infrared | civet walking at night (lower detect threshold; the fast IR walker gives some gaps) |

Each prints its observation sheet + an annotated preview, and saves both to Drive at
`outputs/behavior_showcase/`.

In [ ]:
# 2) Run the showcase
import subprocess, shutil, datetime as dt, imageio_ffmpeg
from base64 import b64encode
from IPython.display import HTML, display
import pandas as pd
from pathlib import Path

FF = imageio_ffmpeg.get_ffmpeg_exe()
CIVET_MODEL    = drive_root / "models" / "civet_bowl_seg_best.pt"   # YOLO-seg civet(+bowl) detector
BEHAVIOR_MODEL = drive_root / "models" / "model.pt"                 # 3-class behavior ResNet
TRAY_BOX       = "706,492,808,573"                                 # Camera-3 DAYTIME food-tray box
DAY_VIDEO      = drive_root / "data" / "raw" / "cam3" / "A03_20260405140327.mp4"
NIGHT_VIDEO    = drive_root / "data" / "raw" / "cam3" / "A03_20260405000000.mp4"
for p in (CIVET_MODEL, BEHAVIOR_MODEL, DAY_VIDEO, NIGHT_VIDEO):
    assert p.exists(), f"Missing on Drive: {p}"

SHOWCASE_DIR = drive_root / "outputs" / "behavior_showcase"          # results persist here
SHOWCASE_DIR.mkdir(parents=True, exist_ok=True)

DAY = "2026-04-05 14:03:27"; NIGHT = "2026-04-05 00:00:00"         # recording start clocks
# (name, video, start_s, end_s, tray_box or None, recording_start, classify_conf)
SHOWCASE = [
    ("Locomotion_day",   DAY_VIDEO,   7120,  7148,  TRAY_BOX, DAY,   0.50),
    ("Eating_day",       DAY_VIDEO,   6345,  6390,  TRAY_BOX, DAY,   0.50),
    ("Resting_day",      DAY_VIDEO,   7640,  7700,  TRAY_BOX, DAY,   0.50),
    ("Eating_night",     NIGHT_VIDEO, 2145,  2190,  None,     NIGHT, 0.50),
    ("Locomotion_night", NIGHT_VIDEO, 11600, 11680, None,     NIGHT, 0.35),  # fast IR walker: lower conf
]

def _show(path, title):
    data = b64encode(Path(path).read_bytes()).decode()
    display(HTML(f"<h3>{title}</h3>"
                 f'<video width="640" controls src="data:video/mp4;base64,{data}"></video>'))

for name, video, a, b, tray, rec_start, cconf in SHOWCASE:
    clip = f"/content/show_{name}.mp4"; outdir = f"/content/show_out_{name}"
    subprocess.run([FF, "-y", "-v", "error", "-ss", str(a), "-to", str(b),
                    "-i", str(video), "-c:v", "libx264", "-pix_fmt", "yuv420p", clip], check=True)
    clk = (dt.datetime.fromisoformat(rec_start) + dt.timedelta(seconds=a)).strftime("%Y-%m-%d %H:%M:%S")
    cmd = ["python", str(PIPE / "run_tracked_behavior.py"), "--video", clip,
           "--civet-model", str(CIVET_MODEL), "--behavior-model", str(BEHAVIOR_MODEL),
           "--out", outdir, "--cctv-start", clk, "--classify-conf", str(cconf)]
    if tray:
        cmd += ["--tray-box", tray]
    subprocess.run(cmd, check=True)
    h264 = f"{outdir}/preview_h264.mp4"
    subprocess.run([FF, "-y", "-v", "error", "-i", f"{outdir}/tracked_preview.mp4",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", h264], check=True)
    shutil.copy2(h264, SHOWCASE_DIR / f"{name}_preview.mp4")
    shutil.copy2(f"{outdir}/observation_sheet.csv", SHOWCASE_DIR / f"{name}_observation_sheet.csv")
    _show(h264, name)
    print(f"{name} — observation sheet (time, status, behavior):")
    display(pd.read_csv(f"{outdir}/observation_sheet.csv"))

print("Saved previews + observation sheets to Drive:", SHOWCASE_DIR)

## 3) Ethogram summary - Duration + Frequency (Phase-1 format)

Two tables per observation, matching the Phase-1 slide:
* **Duration sheet** - chronological rows pairing each Inactive span with the Active span that follows it, plus Total Inactive / Active minutes.
* **Frequency sheet** - **Eating** events per clock hour (single behaviour, as scoped).

The helper functions below mirror `pipeline/ethogram_sheets.py`. `min_span` smooths the
duration sheet by absorbing brief Active/Inactive flips and short not-visible dropouts
into their neighbour (0 = off).

In [ ]:
# 3) Ethogram summary - per showcase clip (generated from each clip's observation sheet)
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML

# --- duration + frequency, mirrors pipeline/ethogram_sheets.py (kept inline so the notebook is self-contained) ---
def _merge_runs(bouts):
    runs = []
    for r in bouts.to_dict("records"):
        if runs and runs[-1]["status"] == r["status"]:
            runs[-1]["end_time"] = r["end_time"]; runs[-1]["duration_s"] += r["duration_s"]
        else:
            runs.append({"status": r["status"], "start_time": r["start_time"],
                         "end_time": r["end_time"], "duration_s": r["duration_s"]})
    return runs

def _coalesce(runs):
    out = []
    for r in runs:
        if out and out[-1]["status"] == r["status"]:
            out[-1]["end_time"] = r["end_time"]; out[-1]["duration_s"] += r["duration_s"]
        else:
            out.append(dict(r))
    return out

def _smooth(runs, min_span):
    if min_span <= 0:
        return runs
    runs = [dict(r) for r in runs]
    while len(runs) > 1:
        cand = None
        for i, r in enumerate(runs):
            if r["duration_s"] >= min_span:
                continue
            neigh = [(j, runs[j]) for j in (i - 1, i + 1)
                     if 0 <= j < len(runs) and runs[j]["status"] in ("Active", "Inactive")]
            if neigh and (cand is None or r["duration_s"] < cand[1]):
                cand = (i, r["duration_s"], neigh)
        if cand is None:
            break
        i, _, neigh = cand
        _, tgt = max(neigh, key=lambda t: t[1]["duration_s"])
        runs[i]["status"] = tgt["status"]; runs = _coalesce(runs)
    return runs

def duration_sheet(bouts, min_span=0):
    runs = _smooth(_merge_runs(bouts), min_span)
    rows, i = [], 0
    while i < len(runs):
        cur = runs[i]
        if cur["status"] == "Unknown":
            i += 1; continue
        row = {"No": len(rows) + 1, "Inactive_start": "", "Inactive_stop": "",
               "Active_start": "", "Active_stop": ""}
        if cur["status"] == "Inactive":
            row["Inactive_start"], row["Inactive_stop"] = cur["start_time"], cur["end_time"]
            nxt = runs[i + 1] if i + 1 < len(runs) else None
            if nxt and nxt["status"] == "Active":
                row["Active_start"], row["Active_stop"] = nxt["start_time"], nxt["end_time"]; i += 1
        else:
            row["Active_start"], row["Active_stop"] = cur["start_time"], cur["end_time"]
        rows.append(row); i += 1
    tot = {"Active": 0, "Inactive": 0}
    for r in runs:
        if r["status"] in tot:
            tot[r["status"]] += r["duration_s"]
    rows.append({"No": "Total", "Inactive_start": f"{tot['Inactive']/60:.1f} min",
                 "Inactive_stop": "", "Active_start": f"{tot['Active']/60:.1f} min", "Active_stop": ""})
    return pd.DataFrame(rows)

def frequency_sheet(bouts, behavior="Eating"):
    def h(t): return datetime.strptime(t, "%H:%M:%S").hour
    sh, eh = h(bouts["start_time"].iloc[0]), h(bouts["end_time"].iloc[-1])
    hours = list(range(sh, eh + 1)) or [sh]
    counts = {x: 0 for x in hours}
    for t in bouts.loc[bouts["behavior"] == behavior, "start_time"]:
        counts[h(t)] = counts.get(h(t), 0) + 1
    lab = {x: f"{x%24:02d}.01-{(x+1)%24:02d}.00" for x in hours}
    rows = [{"Time": lab[x], behavior: counts[x]} for x in hours]
    rows.append({"Time": "Total", behavior: sum(counts.values())})
    return pd.DataFrame(rows)

def show_ethogram(bouts, title, min_span=0):
    display(HTML(f"<b>{title}</b>"))
    print("Duration sheet:"); display(duration_sheet(bouts, min_span))
    print("Frequency sheet (Eating events / hour):"); display(frequency_sheet(bouts))

# --- per showcase clip (uses the observation sheets saved in step 2) ---
for name, *_ in SHOWCASE:
    show_ethogram(pd.read_csv(SHOWCASE_DIR / f"{name}_observation_sheet.csv"), name)

## 4) Longer runs - arc (16 min) & 1 hour

Two continuous end-to-end runs. The **previews load straight from Drive** (not re-rendered here):
the civet box is coloured by status with the yellow tray region drawn. The Duration + Frequency
tables below were produced by `pipeline/ethogram_sheets.py` on each run
(arc smoothed with `--min-span 45`, hour with `--min-span 60`).

In [ ]:
# 4) Arc + 1-hour: Drive previews + their ethogram tables
import pandas as pd
from IPython.display import display, HTML

# Previews hosted on Drive (shared). Shown via the Drive file viewer - no re-rendering.
HOUR_VIDEO_ID = "1MFvhKGrMErqtcy4owISN-5y3xJdksaRm"   # 1-hour preview (15:45-16:45)
ARC_VIDEO_ID  = "1ncXplGUnhRmmSsEjVG0pmfutSn_eRJzO"   # 16-min arc preview (15:58-16:14)

def drive_video(file_id, title):
    display(HTML(
        f"<b>{title}</b><br>"
        f'<iframe src="https://drive.google.com/file/d/{file_id}/preview" '
        f'width="640" height="400" allow="autoplay"></iframe><br>'
        f'<a href="https://drive.google.com/file/d/{file_id}/view" target="_blank">Open in Drive</a>'))

def table(rows):
    return pd.DataFrame(rows)

# ---- Arc (16 min, 15:58-16:14), min-span 45 ----
drive_video(ARC_VIDEO_ID, "Arc preview (16 min, 15:58-16:14)")
arc_duration = table([
    {"No": 1, "Inactive_start": "",         "Inactive_stop": "",         "Active_start": "15:58:00", "Active_stop": "16:10:48"},
    {"No": 2, "Inactive_start": "16:10:48", "Inactive_stop": "16:14:27", "Active_start": "",         "Active_stop": ""},
    {"No": "Total", "Inactive_start": "3.6 min", "Inactive_stop": "", "Active_start": "12.8 min", "Active_stop": ""},
])
arc_frequency = table([
    {"Time": "15.01-16.00", "Eating": 3},
    {"Time": "16.01-17.00", "Eating": 16},
    {"Time": "Total",       "Eating": 19},
])
print("Arc - Duration sheet:"); display(arc_duration)
print("Arc - Frequency sheet (Eating events / hour):"); display(arc_frequency)

# ---- 1 hour (15:45-16:45), min-span 60 ----
drive_video(HOUR_VIDEO_ID, "1-hour preview (15:45-16:45)")
hour_duration = table([
    {"No": 1, "Inactive_start": "",         "Inactive_stop": "",         "Active_start": "15:48:54", "Active_stop": "16:10:48"},
    {"No": 2, "Inactive_start": "16:10:48", "Inactive_stop": "16:18:42", "Active_start": "16:18:42", "Active_stop": "16:24:49"},
    {"No": 3, "Inactive_start": "",         "Inactive_stop": "",         "Active_start": "16:27:15", "Active_stop": "16:40:09"},
    {"No": 4, "Inactive_start": "",         "Inactive_stop": "",         "Active_start": "16:41:56", "Active_stop": "16:43:13"},
    {"No": "Total", "Inactive_start": "7.9 min", "Inactive_stop": "", "Active_start": "42.2 min", "Active_stop": ""},
])
hour_frequency = table([
    {"Time": "15.01-16.00", "Eating": 22},
    {"Time": "16.01-17.00", "Eating": 41},
    {"Time": "Total",       "Eating": 63},
])
print("1-hour - Duration sheet:"); display(hour_duration)
print("1-hour - Frequency sheet (Eating events / hour):"); display(hour_frequency)
print("(1-hour: 9.9 min not-visible, not counted; arc: ~0 min)")